# Traffic Data Preprocessing Pipeline

This notebook performs comprehensive data cleaning and filtering on traffic detection data.

## Pipeline Stages:

1. **Data Loading**: Load parquet files from date range
2. **Lifetime Filtering**: Remove short-lived detections (noise/false positives)
3. **Footpath Zone Filtering**: Remove vehicles in pedestrian-only zones
4. **Crosswalk Zone Filtering**: Remove vehicles moving parallel to crosswalks
5. **Static Object Removal**: Remove stationary vehicles (parked cars, etc.)

## Output:
Clean dataset ready for near-miss analysis

- pedestrian=1
- bicycle=2
- motorcycle=3
- car=4
- escooter=5,
- van=6
- truck=7
- bus=8

In [43]:
%reset -f

In [44]:
# imports
import pandas as pd
import os
from datetime import datetime, timedelta
import numpy as np
import time
from tqdm import tqdm
import gc
import psutil

In [45]:
# Memory Monitoring Utilities
def log_memory(label=""):
    """Quick memory snapshot of current process"""
    process = psutil.Process()
    mem_mb = process.memory_info().rss / 1024 / 1024
    print(f"[MEMORY] {label}: {mem_mb:.1f} MB")
    return mem_mb

def log_df_memory(df, name="DataFrame"):
    """Log DataFrame memory usage"""
    mem_mb = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"[DF MEMORY] {name}: {mem_mb:.1f} MB ({len(df):,} rows)")
    return mem_mb

In [46]:
START_DATE = "2025-06-01"
END_DATE = "2025-06-01"

DATA_DIR = "/home/ubuntu/data/uploads/objects/clean"
CHUNK_SIZE = 500000

In [47]:
def load_data(data_dir, start_date, end_date):
    """
    Load data with optimized dtypes for memory efficiency
    """
    # Define optimal dtypes upfront - saves ~50% memory
    dtypes = {
        'id': 'int32',           # Was int64 → Save 50%
        'label': 'int8',         # Values 1-8 → Save 87.5%
        'pos_x': 'float32',      # Was float64 → Save 50%
        'pos_y': 'float32',
        'pos_z': 'float32',
        'vel': 'float32',
        'vel_x': 'float32',
        'vel_y': 'float32',
        'yaw': 'float32',
        'size_x': 'float32',
        'size_y': 'float32',
        # timestamp stays int64 (needed for precision)
    }
    
    # List to collect dataframes
    dfs = []
    
    # Loop over all folders within date range
    for folder in tqdm(sorted(os.listdir(data_dir)), desc="Loading data"):
        folder_path = os.path.join(data_dir, folder)
        
        # Skip non-folders
        if not os.path.isdir(folder_path):
            continue
        
        if folder.startswith(start_date) or folder.startswith(end_date):
            # Load all parquet files inside the folder
            df_chunk = pd.read_parquet(folder_path)
            
            # Apply dtypes to matching columns
            for col, dtype in dtypes.items():
                if col in df_chunk.columns:
                    df_chunk[col] = df_chunk[col].astype(dtype)
            
            dfs.append(df_chunk)
            del df_chunk  # Clean up immediately
    
    # Combine all into single dataframe
    if dfs:
        log_memory("Before concat")
        df = pd.concat(dfs, ignore_index=True)
        
        # Destroy the list immediately
        del dfs
        gc.collect()
        
        log_memory("After concat + cleanup")
        log_df_memory(df, "Loaded data")
        return df
    else:
        print("No data found for given date range.")
        return pd.DataFrame()

# Usage
df = load_data(DATA_DIR, START_DATE, END_DATE)
print(f"Loaded {len(df)} records from {START_DATE} to {END_DATE}")

Loading data: 100%|██████████| 336/336 [00:01<00:00, 282.11it/s]


[MEMORY] Before concat: 5519.9 MB
[MEMORY] After concat + cleanup: 5221.8 MB
[DF MEMORY] Loaded data: 2273.1 MB (39,074,272 rows)
Loaded 39074272 records from 2025-06-01 to 2025-06-01


In [48]:
len(df['timestamp'].unique())

863760

In [49]:
df.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel
0,2025-06-01 00:00:00.037209034,8748438,4,61.968750,-58.531250,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.010002,0.0,5.386719,0.0,0.0
1,2025-06-01 00:00:00.037209034,9896407,4,6.921875,25.187500,0.099976,4.589844,2.150391,1.769531,0.000000,-0.000000,0.0,5.417969,0.0,0.0
2,2025-06-01 00:00:00.037209034,10035255,4,-20.265625,56.687500,-0.049988,4.191406,1.990234,1.839844,-0.010002,0.010002,0.0,2.285156,-0.0,0.0
3,2025-06-01 00:00:00.037209034,10049521,6,10.726562,20.984375,-0.020004,5.570312,2.240234,1.940430,-0.000000,0.000000,0.0,2.349609,0.0,0.0
4,2025-06-01 00:00:00.037209034,10096097,4,-25.500000,29.359375,0.239990,4.359375,2.099609,1.879883,0.020004,-0.020004,0.0,5.511719,-0.0,0.0


In [50]:
obj_id = 8748438
df_id = df[df["id"] == obj_id].sort_values("timestamp").reset_index(drop=True)
df_id.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel
0,2025-06-01 00:00:00.037209034,8748438,4,61.96875,-58.53125,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.010002,0.0,5.386719,0.0,0.000000
1,2025-06-01 00:00:00.136997938,8748438,4,61.96875,-58.53125,-0.109985,4.269531,2.009766,1.769531,0.000000,-0.000000,0.0,5.386719,0.0,0.000000
2,2025-06-01 00:00:00.229187965,8748438,4,61.96875,-58.53125,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.010002,0.0,5.378906,-0.0,0.000000
3,2025-06-01 00:00:00.329210997,8748438,4,61.96875,-58.56250,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.020004,0.0,5.375000,-0.0,0.078107
4,2025-06-01 00:00:00.443810940,8748438,4,61.96875,-58.53125,-0.109985,4.269531,2.009766,1.769531,0.010002,-0.020004,0.0,5.371094,-0.0,0.007948


In [51]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39074272 entries, 0 to 39074271
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int8          
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(1), int8(1)
memory usage: 2.2 GB


In [52]:
len(df['timestamp'])

39074272

In [53]:
df['timestamp'].duplicated().sum()

np.int64(38210512)

In [54]:
df.reset_index(drop=True, inplace=True)

In [55]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39074272 entries, 0 to 39074271
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int8          
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(1), int8(1)
memory usage: 2.2 GB


### 1. Lifetime Filtering

In [56]:
# Global minimum detection count required per label
global_lifespan_thresholds = {
        1: 30,
        2: 80,
        3: 60,
        4: 90,
        5: 30,
        6: 100,
        7: 100,   
        8: 180
    }

log_memory("Before lifetime filtering")

# Compute lifespan as detection count in full dataset
lifespan = (
    df.groupby(["id", "label"])["timestamp"]
    .count()
    .reset_index(name="lifespan")
)

# Attach thresholds
lifespan["min_required"] = lifespan["label"].map(global_lifespan_thresholds)

# Identify short-lived objects
lifespan["is_outlier"] = lifespan["lifespan"] < lifespan["min_required"]

# Get outlier IDs
short_lived_ids = set(lifespan.loc[lifespan["is_outlier"], "id"].tolist())

# Destroy lifespan DataFrame immediately
del lifespan
gc.collect()

# Drop them from the cleaned dataframe
df = df[~df["id"].isin(short_lived_ids)]

# Logging
print(f"[lifespan filter] Removed {len(short_lived_ids)} short-lived IDs")

# Clean up IDs set
del short_lived_ids
gc.collect()

log_df_memory(df, "After lifetime filter")

[MEMORY] Before lifetime filtering: 5216.8 MB
[lifespan filter] Removed 3157 short-lived IDs
[DF MEMORY] After lifetime filter: 2564.0 MB (38,964,803 rows)


np.float64(2564.0214986801147)

In [57]:
df.reset_index(inplace = True, drop = True)

### 2. Footpath Zone Filtering

In [58]:
footpath_zones = [{"id":"1081","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((-8.6426068 4.1825497, -5.2591289 6.6106291, 2.4873278 -1.8810115, 3.0701297 -4.1885040, -4.8739637 -7.3121410, -5.9190415 -5.1056689, -14.9469623 -8.3614314, -15.4947729 -6.7531838, -6.4707399 -3.4197283, -5.5463181 -0.1021501, -8.6426068 4.1825497))","min_z":-1.5,"max_z":3.5},
{"id":"1082","name":"FalseDetection (Vehicle as Pedestrian)","type":"analytics","vertices":"POLYGON ((-3.3894790 -13.3168241, 11.5404031 -7.8269771, 18.9154074 -17.2223685, 14.9261910 -19.9865761, 21.4404136 -27.6760383, 20.0765345 -28.6872835, 7.1069287 -14.4841637, -11.9112838 -20.7401988, -12.4875105 -18.6473010, -2.5169133 -15.3193791, -3.3894790 -13.3168241))","min_z":-1.5,"max_z":3.5},
{"id":"1083","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((65.0702212 14.5604360, 36.5624449 3.1898636, 35.8594607 -0.3697733, 48.5353798 -17.2319024, 52.8105665 -14.6263596, 49.1754262 -9.8991876, 52.2197357 2.2111523, 68.4919414 9.0673664, 65.0702212 14.5604360))","min_z":-1.5,"max_z":3.5},
{"id":"1084","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((5.9894856 29.2443364, 7.5000886 30.3643575, 15.4508444 22.7984588, 22.8276900 37.1368332, 27.9001775 34.9101554, 20.8424398 19.4127592, 17.0919999 16.0917894, 5.9894856 29.2443364))","min_z":-1.5,"max_z":3.5}]

import geopandas as gpd
from shapely import wkt

zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

In [59]:
def attach_zones_to_objects(df, gdf_zones, how="inner", batch_size=100000):
    """
    Attach zone information to objects via spatial join.
    Handles duplicates when objects span multiple zones.
    """
    columns = df.columns.tolist()
    num_chunks = len(df) // batch_size + 1
    output_chunks = []

    for i in range(num_chunks):
        chunk = df.iloc[i*batch_size : (i+1)*batch_size].copy()

        if len(chunk) == 0:
            continue

        gdf_chunk = gpd.GeoDataFrame(
            chunk,
            geometry=gpd.points_from_xy(chunk["pos_x"], chunk["pos_y"]),
        )

        joined = gpd.sjoin(gdf_chunk, gdf_zones, how=how, predicate="within")
        
        # 🔥 CRITICAL: Drop geometry immediately after join!
        if 'geometry' in joined.columns:
            joined = joined.drop(columns=['geometry'])
        
        # Destroy gdf_chunk right away
        del gdf_chunk

        # Handle empty spatial joins
        if len(joined) == 0:
            if how == "left":
                chunk["zone"] = np.nan
                output_chunks.append(chunk)
            del joined  # Clean up
            continue

        # Rename columns
        joined = joined.rename(columns={
            "id_left": "id",
            "id_right": "zone"
        })

        # Remove duplicates (objects in multiple zones - keep first)
        joined = joined.drop_duplicates(subset=['id', 'timestamp'], keep='first')
        
        # Select only needed columns
        joined = joined[columns + ["zone"]]
        
        # Convert zone to category dtype (saves memory)
        joined['zone'] = joined['zone'].astype('category')

        output_chunks.append(joined)
        
        # Clean up this iteration
        del joined
        
        # Force garbage collection every 5 batches
        if i % 5 == 0:
            gc.collect()

    # Handle case where no objects are in zones
    if len(output_chunks) == 0:
        result = df.copy()
        result["zone"] = np.nan
        return result

    result = pd.concat(output_chunks, ignore_index=True)
    
    # Destroy output_chunks
    del output_chunks
    gc.collect()
    
    return result

In [60]:
# Attach footpath zones in chunks
total_rows = len(df)
processed_chunks = []

print(f"Attaching footpath zones to {total_rows} rows...")
log_memory("Before footpath zones")

for i in tqdm(range(0, total_rows, CHUNK_SIZE), desc="Attaching footpath zones"):
    chunk = df.iloc[i:i+CHUNK_SIZE].copy()
    processed_chunks.append(attach_zones_to_objects(chunk, gdf_zones, how="left"))
    del chunk  # Clean up chunk

df = pd.concat(processed_chunks, ignore_index=True)

# Destroy the list immediately
del processed_chunks
gc.collect()

log_memory("After footpath zones")
print(f"✓ Zones attached! Total rows: {len(df)}")

Attaching footpath zones to 38964803 rows...
[MEMORY] Before footpath zones: 7445.4 MB


Attaching footpath zones:  27%|██▋       | 21/78 [00:10<00:27,  2.07it/s]


KeyboardInterrupt: 

In [ ]:
def apply_footpath_zone_filter(df):
    """
    Remove vehicles that shouldn't be in footpath zones based on:
    1. Speed limits per vehicle type
    2. Forbidden vehicle types (trucks, buses)
    Memory-optimized with aggressive cleanup.
    """
    max_speed = {
        1: 4.0,   # pedestrian
        2: 6.0,   # bicycle
        3: 12.0,  # motorcycle
        4: 12.0,  # car
        5: 4.0,   # escooter
        6: 12.0,  # van
        7: 0.0,   # truck - forbidden
        8: 0.0,   # bus - forbidden
    }

    df_zone = df[df["zone"].notnull()].copy()
    speed_limit_series = df_zone["label"].map(max_speed)

    # Vehicles that shouldn't be in footpath zones
    forbidden_mask = df_zone["label"].isin([3, 4, 5, 6, 7, 8])

    # Vehicles exceeding speed limits
    speed_exceed_mask = df_zone["vel"] > speed_limit_series

    remove_mask = forbidden_mask | speed_exceed_mask

    removed_ids = df_zone.loc[remove_mask, "id"].unique()
    
    # Clean up intermediate variables immediately
    del df_zone, speed_limit_series, forbidden_mask, speed_exceed_mask, remove_mask
    
    df = df.loc[~df["id"].isin(removed_ids)].copy()

    print(f"[footpath zone] Removed {len(removed_ids)} objects")
    
    # Clean up IDs
    del removed_ids
    gc.collect()

    return df

In [ ]:
# Apply footpath zone filter
df = apply_footpath_zone_filter(df)

[footpath zone] Removed 703 objects


In [ ]:
# Clean up: drop zone column and reset index
df = df.drop(columns=["zone"])
df = df.reset_index(drop=True)

# Force cleanup after dropping geometry-related column
gc.collect()
log_memory("After footpath filter cleanup")

[MEMORY] After footpath filter cleanup: 6896.0 MB


6895.953125

In [ ]:
# Verify data after footpath filtering
print(f"Remaining objects: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")

Remaining objects: 38289116
Unique IDs: 52737


### 3. Crosswalk Zone Filtering

In [ ]:
crosswalk_zones = [
    {"id":"1015","name":"Crosswalk Houba - South","type":"analytics","vertices":"POLYGON ((25.0 -23.6, 42.8 -8.5, 40.3 -5.5, 22.6 -21.0, 25.0 -23.6))","min_z":-1.5,"max_z":3.5},
    {"id":"1016","name":"Crosswalk Amandiers","type":"analytics","vertices":"POLYGON ((-2.7 -13.4, -4.8 -7.4, -1.4 -6.0, 0.7 -12.2, -2.7 -13.4))","min_z":-1.5,"max_z":3.5},
    {"id":"1017","name":"Crosswalk Houba - North","type":"analytics","vertices":"POLYGON ((-3.1 4.7, 13.8 19.3, 16.9 16.1, 0.0 1.4, -3.1 4.7))","min_z":-1.5,"max_z":3.5},
    {"id":"1018","name":"Crosswalk Magnolias","type":"analytics","vertices":"POLYGON ((30.5 15.5, 32.2 18.9, 23.4 23.5, 21.6 20.2, 30.5 15.5))","min_z":-1.5,"max_z":3.5},
    {"id":"1019","name":"Crosswalk Charlotte [1]","type":"analytics","vertices":"POLYGON ((36.4 15.3, 40.4 5.2, 37.1 3.8, 33.1 14.0, 36.4 15.3))","min_z":-1.5,"max_z":3.5}
]

zones_df = pd.DataFrame(crosswalk_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

In [ ]:
def compute_polygon_orientation(poly):
    """
    Calculate the orientation of a polygon based on its longest edge.
    Returns angle in degrees.
    """
    coords = list(poly.exterior.coords)
    edges = []

    for (x1, y1), (x2, y2) in zip(coords, coords[1:]):
        dx, dy = x2 - x1, y2 - y1
        length = (dx**2 + dy**2) ** 0.5
        edges.append((length, dx, dy))

    # Get longest edge
    _, dx, dy = max(edges, key=lambda e: e[0])
    return np.degrees(np.arctan2(dy, dx))

In [ ]:
def filter_parallel_vehicles(df_zone, orientation_deg, threshold=4.0):
    """
    Filter vehicles moving parallel to crosswalk axis.
    Vehicles traveling along the crosswalk (not crossing) are removed.
    
    Args:
        df_zone: DataFrame with objects in a single zone
        orientation_deg: Crosswalk orientation in degrees
        threshold: Maximum angle difference (degrees) to consider parallel
    
    Returns:
        parallel_ids: List of IDs to remove
        df_zone_filtered: Filtered DataFrame
    """
    # Filter all vehicle types (exclude pedestrians)
    vehicle_labels = [3, 4, 6, 7, 8]  # from motorcycle through bus except for the escooter
    vehicles = df_zone[df_zone["label"].isin(vehicle_labels)].copy()

    if vehicles.empty:
        return [], df_zone

    # Compute vehicle heading from yaw
    vehicles["heading_deg"] = np.degrees(vehicles["yaw"])

    # Fallback: use velocity direction if yaw is missing
    missing = vehicles["heading_deg"].isna()
    vehicles.loc[missing, "heading_deg"] = np.degrees(
        np.arctan2(vehicles.loc[missing, "vel_y"], vehicles.loc[missing, "vel_x"])
    )

    # Calculate angular difference
    def angle_diff(a, b):
        d = (a - b + 180) % 360 - 180
        return abs(d)

    vehicles["angle_diff"] = vehicles.apply(
        lambda r: angle_diff(r["heading_deg"], orientation_deg),
        axis=1
    )

    # Find vehicles moving parallel to crosswalk
    parallel = vehicles[vehicles["angle_diff"] < threshold]["id"].unique().tolist()

    # Remove parallel vehicles
    df_zone_filtered = df_zone[~df_zone["id"].isin(parallel)]

    return parallel, df_zone_filtered

In [ ]:
# Precompute zone orientations
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

removed_ids_global = []
processed_chunks = []

# Attach crosswalk zones in chunks
total_rows = len(df)
print(f"Attaching crosswalk zones to {total_rows} rows...")
log_memory("Before crosswalk zones")

for i in tqdm(range(0, total_rows, CHUNK_SIZE), desc="Attaching crosswalk zones"):
    chunk = df.iloc[i:i+CHUNK_SIZE].copy()
    processed_chunk = attach_zones_to_objects(chunk, gdf_zones, how="left")
    processed_chunks.append(processed_chunk)
    del chunk  # Clean up chunk

df = pd.concat(processed_chunks, ignore_index=True)

# Destroy chunks immediately
del processed_chunks
gc.collect()

log_memory("After crosswalk zone attachment")
print(f"✓ Zones attached! Total rows: {len(df)}")

# Check zone distribution
print(f"Objects in zones: {df['zone'].notna().sum()}")
print(f"Objects outside zones: {df['zone'].isna().sum()}")

Attaching crosswalk zones to 38289116 rows...
[MEMORY] Before crosswalk zones: 6896.1 MB


Attaching crosswalk zones: 100%|██████████| 77/77 [00:34<00:00,  2.24it/s]


[MEMORY] After crosswalk zone attachment: 8255.3 MB
✓ Zones attached! Total rows: 38289116
Objects in zones: 792138
Objects outside zones: 37496978


In [ ]:
# Process each crosswalk zone separately
for zone_id, zone_df in tqdm(df.groupby("zone"), desc="Filtering parallel vehicles"):
    # Skip NaN zones (objects outside all crosswalk zones)
    if pd.isna(zone_id):
        print(f"Skipping {len(zone_df)} objects outside all crosswalk zones")
        continue
    
    print(f"Processing crosswalk zone {zone_id} with {len(zone_df)} objects")

    zone_orientation = float(
        gdf_zones.loc[gdf_zones["id"] == str(zone_id), "orientation_deg"].iloc[0]
    )

    removed_ids, zone_filtered = filter_parallel_vehicles(
        zone_df, orientation_deg=zone_orientation, threshold=4.0
    )

    removed_ids_global.append(removed_ids)
    print(f"  → Removed {len(removed_ids)} parallel vehicles")
    
    # Clean up iteration variables
    del removed_ids, zone_filtered

# Flatten list and remove IDs
removed_ids_global = [item for sublist in removed_ids_global for item in sublist]
df = df[~df["id"].isin(removed_ids_global)]

print(f"\n✓ Total parallel vehicles removed: {len(removed_ids_global)}")
print(f"✓ Remaining objects: {len(df)}")

# Clean up IDs list
del removed_ids_global
gc.collect()

log_memory("After crosswalk filtering")

Filtering parallel vehicles:   0%|          | 0/5 [00:00<?, ?it/s]

Processing crosswalk zone 1015 with 265019 objects


Filtering parallel vehicles:  20%|██        | 1/5 [00:02<00:10,  2.60s/it]

  → Removed 2 parallel vehicles
Processing crosswalk zone 1016 with 71078 objects
  → Removed 1 parallel vehicles
Processing crosswalk zone 1017 with 254382 objects


Filtering parallel vehicles:  60%|██████    | 3/5 [00:03<00:01,  1.12it/s]

  → Removed 1 parallel vehicles
Processing crosswalk zone 1018 with 99796 objects
  → Removed 0 parallel vehicles
Processing crosswalk zone 1019 with 101863 objects


Filtering parallel vehicles: 100%|██████████| 5/5 [00:03<00:00,  1.47it/s]

  → Removed 0 parallel vehicles



✓ Total parallel vehicles removed: 4
✓ Remaining objects: 38287948
[MEMORY] After crosswalk filtering: 10709.7 MB


10709.74609375

In [ ]:
# Clean up: drop zone column and reset index
df = df.drop(columns=["zone"])
df = df.reset_index(drop=True)

# Force cleanup after dropping zone column
gc.collect()
log_memory("After crosswalk filter cleanup")

[MEMORY] After crosswalk filter cleanup: 10417.6 MB


10417.6328125

In [ ]:
# Verify data after crosswalk filtering
print(f"Remaining objects: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")

Remaining objects: 38287948
Unique IDs: 52733


### 4. Remove Static Objects

In [ ]:
STATIC_THRESHOLD = 0.5     # m/s - velocity below this is considered static
STATIC_RATIO_MIN = 0.8     # 80% of lifetime must be static

log_memory("Before static filter")

# Build per-object velocity history
df_vel = (
    df.groupby(["id", "label"])["vel"]
    .apply(list)
    .reset_index()
)

# Compute lifespan
df_vel["lifespan"] = df_vel["vel"].apply(len)

# Count static frames
df_vel["static_frames"] = df_vel["vel"].apply(
    lambda v: sum(vi < STATIC_THRESHOLD for vi in v)
)

# Calculate static ratio
df_vel["static_ratio"] = df_vel["static_frames"] / df_vel["lifespan"]

# Flag static objects
df_vel["is_static"] = df_vel["static_ratio"] >= STATIC_RATIO_MIN

# Get IDs to remove
removable_static_ids = set(
    df_vel[df_vel["is_static"]]["id"].astype(int).tolist()
)

print(f"[static filter] Found {len(removable_static_ids)} static objects")

# Destroy df_vel immediately
del df_vel
gc.collect()

# Remove static objects
df = df[~df['id'].isin(removable_static_ids)]

print(f"[static filter] Removed {len(removable_static_ids)} static objects")
print(f"Remaining objects: {len(df)}")

# Clean up IDs set
del removable_static_ids
gc.collect()

log_memory("After static filter")
log_df_memory(df, "After static filter")

[MEMORY] Before static filter: 10417.6 MB


[static filter] Found 6672 static objects
[static filter] Removed 6672 static objects
Remaining objects: 14994565
[MEMORY] After static filter: 9060.3 MB
[DF MEMORY] After static filter: 986.7 MB (14,994,565 rows)


np.float64(986.6952753067017)

In [ ]:
# Final cleanup
df = df.reset_index(drop=True)
gc.collect()
log_memory("After final cleanup")

[MEMORY] After final cleanup: 9060.3 MB


9060.3203125

In [ ]:
# Final verification
print("="*60)
print("FINAL DATASET SUMMARY")
print("="*60)
print(f"Total objects: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")
print(f"Unique timestamps: {df['timestamp'].nunique()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())
print("="*60)

FINAL DATASET SUMMARY
Total objects: 14994565
Unique IDs: 46061
Unique timestamps: 846344

Label distribution:
label
1    4716668
2     172748
3     146746
4    8928679
5      63197
6     674627
7      92015
8     199885
Name: count, dtype: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14994565 entries, 0 to 14994564
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int8          
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(1), int8(1)
memory usage: 872.3 MB


In [ ]:
df.reset_index(inplace = True, drop = True)

In [ ]:
# =============================================================================
# DATA QUALITY FILTERS (Abnormal Acc. & Jitter) - Run on df
# =============================================================================

print(f"Initial rows: {len(df):,}")

df = df.sort_values(['id', 'timestamp'])

# Abnormal Acc. FILTER
df['_dt'] = df.groupby('id')['timestamp'].diff().dt.total_seconds()
df['_dv'] = df.groupby('id')['vel'].diff()
df['_acc'] = (df['_dv'] / df['_dt']).fillna(0).abs()
teleport_mask = df['_acc'] > 10.0
print(f"-> Abnormal Acceleration glitches: {teleport_mask.sum():,}")

# JITTER FILTER
df['_prev_yaw'] = df.groupby('id')['yaw'].shift(1)
yaw_diff = (df['yaw'] - df['_prev_yaw']).fillna(0).abs()
yaw_diff = np.where(yaw_diff > np.pi, 2*np.pi - yaw_diff, yaw_diff)
jitter_mask = (yaw_diff > 0.785) & (df['vel'] > 2.0)
print(f"-> Jitter glitches: {jitter_mask.sum():,}")

# APPLY
bad_rows = teleport_mask | jitter_mask
df = df[~bad_rows].copy()
df.drop(columns=['_dt', '_dv', '_acc', '_prev_yaw'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Removed {bad_rows.sum():,} rows. Remaining: {len(df):,}")
gc.collect()

Initial rows: 14,994,565


-> Abnormal Acceleration glitches: 49,460
-> Jitter glitches: 18,507
Removed 67,062 rows. Remaining: 14,927,503


17

In [ ]:
# =============================================================================
# FILTER 1: GHOST VEHICLE REMOVAL (Simplest - O(n))
# =============================================================================
# Remove vehicles that spawn/despawn inside the detection zone
# (tracking errors, partial occlusions, brief appearances)

print("\n" + "="*70)
print("FILTER 1: Ghost Vehicle Removal")
print("="*70)

from filters.ghost_filter import filter_ghost_vehicles

print(f"Before ghost filter: {len(df):,} rows, {df['id'].nunique():,} unique vehicles")
log_memory("Before ghost filter")

df = filter_ghost_vehicles(df, verbose=True)
df.reset_index(drop=True, inplace=True)

print(f"After ghost filter: {len(df):,} rows, {df['id'].nunique():,} unique vehicles")
log_memory("After ghost filter")
gc.collect()
print("="*70)


FILTER 1: Ghost Vehicle Removal
Before ghost filter: 14,927,503 rows, 46,061 unique vehicles
[MEMORY] Before ghost filter: 9105.2 MB

GHOST VEHICLE FILTER
Input vehicles: 46,061


Checking spawn/despawn locations...


Checking 46,061 vehicles: 100%|██████████| 46061/46061 [00:00<00:00, 118126.54vehicle/s]



  Ghost vehicles detected: 5,988
  - Breakdown:
    Spawned inside: 2,480
    Despawned inside: 2,423
    Both (full ghosts): 1,085

  Vehicles after filtering: 40,073
  Records removed: 1,537,631
  Percentage removed: 10.30%
After ghost filter: 13,389,872 rows, 40,073 unique vehicles
[MEMORY] After ghost filter: 9066.2 MB


In [ ]:
len(df)

13389872

# Data extraction - Near miss pairs (Heavy - Heavy)

*Current zone data, pedastrian crossing zones, is for Light to Heavy vehicles only!

In [ ]:
# Lane specific zones for detection
lane_zones = [
    {"id": "1085", "name": "A-L1", "type": "detection", 
     "vertices": "POLYGON ((24.134 23.325, 27.775 21.504, 49.243 62.543, 45.222 64.211, 45.222 64.211, 24.134 23.325))", 
     "min_z": -1.5, "max_z": 3.5},

    {"id": "1086", "name": "A-L2", "type": "detection", 
     "vertices": "POLYGON ((27.906 21.434, 31.918 19.361, 53.589 60.605, 49.425 62.464, 49.425 62.464, 27.906 21.434))", 
     "min_z": -1.5, "max_z": 3.5},
    
    {"id": "1087", "name": "B-L1", "type": "detection", 
     "vertices": "POLYGON ((36.365 15.278, 39.017 9.579, 88.804 27.818, 86.184 32.753, 36.365 15.278))", 
     "min_z": -1.5, "max_z": 3.5},
    
    {"id": "1088", "name": "B-L2", "type": "detection", 
     "vertices": "POLYGON ((38.985 9.247, 89.027 27.723, 90.253 25.348, 60.751 12.934, 40.57 5.279, 38.985 9.247))", 
     "min_z": -1.5, "max_z": 3.5},
    
    {"id": "1089", "name": "C-L1", "type": "detection", 
     "vertices": "POLYGON ((35.542 -17.295, 43.74 -10.167, 46.858 -13.553, 52.472 -24.245, 84.044 -60.732, 78.756 -66.262, 35.542 -17.295))", 
     "min_z": -1.5, "max_z": 3.5},
    
    {"id": "1090", "name": "C-L2", "type": "detection", 
     "vertices": "POLYGON ((35.421 -17.507, 32.033 -20.757, 74.427 -68.683, 77.985 -65.481, 35.421 -17.507))", 
     "min_z": -1.5, "max_z": 3.5},
    
    {"id": "1091", "name": "D-L1", "type": "detection", 
     "vertices": "POLYGON ((-3.559 -10.559, -2.647 -12.84, -40.175 -26.861, -41.008 -24.361, -3.559 -10.559))", 
     "min_z": -1.5, "max_z": 3.5},
    
    {"id": "1092", "name": "D-L2", "type": "detection", 
     "vertices": "POLYGON ((-3.744 -10.457, -41.152 -24.168, -42.046 -21.833, -4.887 -7.824, -3.744 -10.457))", 
     "min_z": -1.5, "max_z": 3.5},
    
    {"id": "1093", "name": "E-L2", "type": "detection", 
     "vertices": "POLYGON ((7.97 14.272, 11.903 17.612, -24.514 57.166, -27.924 54.162, 7.97 14.272))", 
     "min_z": -1.5, "max_z": 3.5},
    
    {"id": "1094", "name": "E-L1", "type": "detection", 
     "vertices": "POLYGON ((-39.209 44.825, -2.024 4.96, 5.445 11.618, -9.819 28.018, -19.074 31.104, -20.536 31.997, -35.881 47.829, -39.209 44.825))", 
     "min_z": -1.5, "max_z": 3.5},
]

In [ ]:
# =============================================================================
# ZONE ASSIGNMENT - Spatial Join
# =============================================================================
import sys
sys.path.append('/home/ubuntu/prem')

from ssm.utils import assign_zones_to_vehicles

print("="*70)
print("ZONE ASSIGNMENT")
print("="*70)

# Assign zones to all vehicles based on position
log_memory("Before zone assignment")
df = assign_zones_to_vehicles(df, lane_zones)
log_memory("After zone assignment")

# Zone statistics
zone_counts = df['zone'].value_counts()
print(zone_counts)
print(f"\nTotal vehicles in lanes: {(df['zone'] != 'unknown').sum():,}")
print(f"Vehicles outside lanes: {(df['zone'] == 'unknown').sum():,}")
print("="*70)

[SSM] Numba enabled with 7 threads
ZONE ASSIGNMENT
[MEMORY] Before zone assignment: 9066.3 MB

Assigning zones to 13,389,872 vehicles using spatial join...
Processing in batches of 100,000 rows


Zone assignment: 100%|██████████| 134/134 [00:18<00:00,  7.18it/s]



✓ Zone assignment complete!
  Vehicles in zones: 7,170,937
  Vehicles outside zones: 6,218,935
[MEMORY] After zone assignment: 9358.6 MB
zone
unknown    6218935
E-L1       1750667
C-L1       1646451
B-L1       1122993
E-L2        646811
A-L1        624816
D-L1        493823
C-L2        468441
B-L2        259260
A-L2        100562
D-L2         57113
Name: count, dtype: int64

Total vehicles in lanes: 7,170,937
Vehicles outside lanes: 6,218,935


---
# Near-Miss Detection Using SSM Methods

Apply Surrogate Safety Measures to detect vehicle-vehicle near-miss events with zone information.

**Optimized Workflow:**
1. Generate base pairs ONCE using `find_all_nearby_pairs()`
2. Apply method-specific filters: M-DRAC (longitudinal) and SPF (all types)
3. Detect conflicts from filtered pairs

**Methods:**
- **M-DRAC** (Modified DRAC): Longitudinal car-following conflicts (rear-end)
- **SPF** (Safety Potential Field): All conflict types (crossing, merging, head-on, etc.)

**Output Schema:**
- M-DRAC: `timestamp, pair_id, zone, conflict_type, interaction, distance, ttc, closing_speed, speed_diff, mdrac, severity`
- SPF: `timestamp, pair_id, zone, conflict_type, interaction, distance, ttc, closing_speed, o_field, s_field, composite_risk, severity`

---

In [ ]:
# =============================================================================
# STEP 1: GENERATE BASE PAIRS
# =============================================================================
print("\n" + "="*70)
print("STEP 1: Generate Base Pairs (Once)")
print("="*70)

from ssm.utils import find_all_nearby_pairs, get_mdrac_pairs, get_spf_pairs, load_config
from ssm.m_drac import ModifiedDRAC
from ssm.spf import SafetyPotentialField
import time

# Load config and initialize detectors
config = load_config()
mdrac_detector = ModifiedDRAC(config)
spf_detector = SafetyPotentialField(config)

# Filter for lane vehicles only (exclude 'unknown' zone)
df_lanes = df[df['zone'] != 'unknown'].copy()
print(f"\nFiltered to {len(df_lanes):,} records in lanes (excluded {(df['zone'] == 'unknown').sum():,} outside lanes)")
print(f"Unique vehicles in lanes: {df_lanes['id'].nunique():,}")

# Generate all nearby pairs ONCE (expensive O(n²) operation)
print("\nGenerating base pairs from lane vehicles...")
log_memory("Before pair generation")
t_start = time.time()

base_pairs = find_all_nearby_pairs(df_lanes, config)

t_pairs = time.time() - t_start
log_memory("After pair generation")

print(f"\n✓ Generated {len(base_pairs):,} base pairs in {t_pairs:.2f}s")
print(f"  Memory: {base_pairs.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print("="*70)


STEP 1: Generate Base Pairs (Once)

Filtered to 7,170,937 records in lanes (excluded 6,218,935 outside lanes)
Unique vehicles in lanes: 25,167

Generating base pairs from lane vehicles...
[MEMORY] Before pair generation: 9686.9 MB
  Filtered vehicles: 3,170,244
  Generating pairs (max_distance=8.0m)...
  Processing 718,458 timestamps (batch_size=20,000)


  Pair generation: 100%|██████████| 36/36 [00:02<00:00, 12.41it/s]


  Applying overlap filter (buffer=0.1m)...
  ✓ Generated 202,188 nearby pairs (after overlap filter)
[MEMORY] After pair generation: 9687.4 MB

✓ Generated 202,188 base pairs in 3.58s
  Memory: 18.1 MB


In [ ]:
# =============================================================================
# STEP 2: M-DRAC NEAR-MISS DETECTION
# =============================================================================
print("\n" + "="*70)
print("STEP 3: M-DRAC (Modified DRAC) Near-Miss Detection")
print("="*70)

print("\nApplying M-DRAC filters to base pairs...")
print("(Filters: same-lane, approaching, follower faster than leader)")

# Apply M-DRAC-specific filters to base pairs
log_memory("Before M-DRAC filtering")
t_start = time.time()

mdrac_pairs = get_mdrac_pairs(base_pairs, config, skip_pair_generation=True)

t_mdrac_filter = time.time() - t_start
print(f"✓ M-DRAC pairs filtered in {t_mdrac_filter:.2f}s")

# Detect conflicts from filtered pairs
print("\nDetecting M-DRAC conflicts...")
log_memory("Before M-DRAC detection")
t_start = time.time()

mdrac_conflicts = mdrac_detector.detect(mdrac_pairs, is_pairs_data=True)

t_mdrac_detect = time.time() - t_start
log_memory("After M-DRAC detection")

# Display results
print(f"\n{'='*70}")
print(f"M-DRAC Results: {len(mdrac_conflicts):,} conflicts detected")
print(f"  Filtering time: {t_mdrac_filter:.2f}s")
print(f"  Detection time: {t_mdrac_detect:.2f}s")
print(f"  Total M-DRAC time: {t_mdrac_filter + t_mdrac_detect:.2f}s")
print(f"{'='*70}")

# if len(mdrac_conflicts) > 0:
#     print("\nZone Distribution:")
#     print(mdrac_conflicts['zone'].value_counts())
    
#     print("\nSeverity Distribution:")
#     print(mdrac_conflicts['severity'].value_counts().to_dict())
    
#     print("\nTop 5 Most Critical (highest MDRAC):")
#     print(mdrac_conflicts.nlargest(5, 'mdrac')[['timestamp', 'pair_id', 'zone', 
#                                                   'distance', 'ttc', 'mdrac', 'severity']])
    
#     print("\nOutput Schema:")
#     print(mdrac_conflicts.columns.tolist())
# else:
#     print("\n⚠ No M-DRAC conflicts detected in this dataset")


STEP 3: M-DRAC (Modified DRAC) Near-Miss Detection

Applying M-DRAC filters to base pairs...
(Filters: same-lane, approaching, follower faster than leader)
[MEMORY] Before M-DRAC filtering: 9687.4 MB


 Approaching pairs: 101,560
  Same-lane pairs (lat <= 2.0m): 10,634
  Speed diff > 0.5: 5,559
  Final MDRAC pairs: 125
✓ M-DRAC pairs filtered in 0.56s

Detecting M-DRAC conflicts...
[MEMORY] Before M-DRAC detection: 9688.5 MB
 Approaching pairs: 125
  Same-lane pairs (lat <= 2.0m): 125
  Speed diff > 0.5: 125
  Final MDRAC pairs: 125
[MEMORY] After M-DRAC detection: 9688.8 MB

M-DRAC Results: 6 conflicts detected
  Filtering time: 0.56s
  Detection time: 0.03s
  Total M-DRAC time: 0.59s


In [ ]:
# =============================================================================
# SAVE M-DRAC RESULTS
# =============================================================================

if len(mdrac_conflicts) > 0:
    output_path = 'results/mdrac_avg_conflicts_new.csv'
    mdrac_conflicts.to_csv(output_path, index=False)
    print(f"\n✓ Saved {len(mdrac_conflicts):,} M-DRAC conflicts to {output_path}")
    
    # Show sample
    print("\nSample output (first 3 rows):")
    print(mdrac_conflicts.head(3).to_string())
    
    # Memory cleanup
    del mdrac_conflicts
    gc.collect()
else:
    print("\n⚠ No M-DRAC conflicts to save")


✓ Saved 6 M-DRAC conflicts to results/mdrac_avg_conflicts_new.csv

Sample output (first 3 rows):
                      timestamp       id1       id2 interaction    leader      dist       TTC      MDRAC  closing_speed  speed_diff    yaw_diff                                                                           link
0 2025-06-01 09:04:00.123322010  10518668  10518739   car_v_car  10518739  4.958519  0.425021  13.724673      11.666538    7.329702   85.607948  https://di-india-collab-2.flow-analytics.io/tools/replay/2025-06-01T09:04:00Z
1 2025-06-01 09:35:30.561595917  10540034  10540130   car_v_car  10540130  4.248736  0.341212  18.246611      12.451910    7.358582  159.046143  https://di-india-collab-2.flow-analytics.io/tools/replay/2025-06-01T09:35:30Z
2 2025-06-01 14:30:41.212084055  10765669  10765702   car_v_car  10765702  6.171776  0.945711  32.630344       6.526069    3.008981  177.538574  https://di-india-collab-2.flow-analytics.io/tools/replay/2025-06-01T14:30:41Z


In [86]:
# =============================================================================
# STEP 3: SPF NEAR-MISS DETECTION
# =============================================================================
print("\n" + "="*70)
print("STEP 2: SPF (Safety Potential Field) Near-Miss Detection")
print("="*70)

print("\nApplying SPF filters to base pairs...")
print("(Filters: approaching vehicles, all conflict types)")

# Apply SPF-specific filters to base pairs
log_memory("Before SPF filtering")
t_start = time.time()

spf_pairs = get_spf_pairs(base_pairs, config, skip_pair_generation=True)

t_spf_filter = time.time() - t_start
print(f"✓ SPF pairs filtered in {t_spf_filter:.2f}s")

# Detect conflicts from filtered pairs
print("\nDetecting SPF conflicts...")
log_memory("Before SPF detection")
t_start = time.time()

spf_conflicts = spf_detector.detect(spf_pairs, is_pairs_data=True)

t_spf_detect = time.time() - t_start
log_memory("After SPF detection")

# Display results
print(f"\n{'='*70}")
print(f"SPF Results: {len(spf_conflicts):,} conflicts detected")
print(f"  Filtering time: {t_spf_filter:.2f}s")
print(f"  Detection time: {t_spf_detect:.2f}s")
print(f"  Total SPF time: {t_spf_filter + t_spf_detect:.2f}s")
print(f"{'='*70}")

# if len(spf_conflicts) > 0:
#     print("\nZone Distribution:")
#     print(spf_conflicts['zone'].value_counts())
    
#     print("\nConflict Type Distribution:")
#     print(spf_conflicts['conflict_type'].value_counts().to_dict())
    
#     print("\nSeverity Distribution:")
#     print(spf_conflicts['severity'].value_counts().to_dict())
    
#     print("\nTop 5 Most Critical (highest composite risk):")
#     print(spf_conflicts.nlargest(5, 'composite_risk')[['timestamp', 'pair_id', 'zone',
#                                                         'conflict_type', 'distance', 'ttc', 
#                                                         'composite_risk', 'severity']])
    
#     print("\nOutput Schema:")
#     print(spf_conflicts.columns.tolist())
# else:
#     print("\n⚠ No SPF conflicts detected in this dataset")


STEP 2: SPF (Safety Potential Field) Near-Miss Detection

Applying SPF filters to base pairs...
(Filters: approaching vehicles, all conflict types)
[MEMORY] Before SPF filtering: 9000.1 MB
 Approaching pairs: 34,164
  Final SPF pairs: 34,164
✓ SPF pairs filtered in 0.01s

Detecting SPF conflicts...
[MEMORY] Before SPF detection: 9000.1 MB
 Approaching pairs: 34,164
  Final SPF pairs: 34,164
  Processing 34,164 pairs in batches of 50,000...


  SPF calculation: 100%|██████████| 1/1 [00:00<00:00, 53.72it/s]


  ✓ 982 conflicts detected
[MEMORY] After SPF detection: 9012.2 MB

SPF Results: 982 conflicts detected
  Filtering time: 0.01s
  Detection time: 0.12s
  Total SPF time: 0.12s


In [87]:
# =============================================================================
# SAVE SPF RESULTS
# =============================================================================

if len(spf_conflicts) > 0:
    output_path = 'results/spf_conflicts.csv'
    spf_conflicts.to_csv(output_path, index=False)
    print(f"\n✓ Saved {len(spf_conflicts):,} SPF conflicts to {output_path}")
    
    # Show sample
    print("\nSample output (first 3 rows):")
    print(spf_conflicts.head(3).to_string())
    
    # Memory cleanup
    del spf_conflicts
    gc.collect()
else:
    print("\n⚠ No SPF conflicts to save")


✓ Saved 982 SPF conflicts to results/spf_conflicts.csv

Sample output (first 3 rows):
                      timestamp       id1       id2 interaction      dist       TTC  composite_risk  closing_speed  speed_diff  yaw_diff                                                                         link
0 2025-06-01 00:07:50.832639933  10287921  10288098   car_v_car  3.781766  0.914998        0.969074       4.133087    3.848074  2.461928  https://di-india-collab.flow-analytics.io/tools/replay/2025-06-01T00:07:40Z
1 2025-06-01 00:07:50.941036940  10287921  10288098   car_v_car  3.313090  0.762758        0.989730       4.343566    3.944590  5.371479  https://di-india-collab.flow-analytics.io/tools/replay/2025-06-01T00:07:40Z
2 2025-06-01 00:07:51.038043976  10287921  10288098   car_v_car  2.912292  0.642655        0.993468       4.531657    4.047575  6.266726  https://di-india-collab.flow-analytics.io/tools/replay/2025-06-01T00:07:41Z


---
# Analysis Complete! ✅

**Outputs Generated:**
- `results/mdrac_conflicts.csv` - Longitudinal car-following conflicts (rear-end)
  - Schema: timestamp, pair_id, zone, conflict_type, interaction, distance, ttc, closing_speed, speed_diff, mdrac, severity
  
- `results/spf_conflicts.csv` - General conflicts (all geometry types)
  - Schema: timestamp, pair_id, zone, conflict_type, interaction, distance, ttc, closing_speed, o_field, s_field, composite_risk, severity

**Zone Information:**
- Each conflict is tagged with the lane/zone where it occurred
- Cross-lane SPF conflicts show combined zones (e.g., "A-L1_C-L2")
- Same-lane conflicts show single zone (e.g., "A-L1")

**Next Steps:**
1. Visualize conflicts using `plotter.py`
2. Analyze zone-specific conflict patterns
3. Identify high-risk vehicle pairs
4. Generate temporal/spatial heatmaps

---

In [88]:
# =============================================================================
# PERFORMANCE SUMMARY
# =============================================================================
print("\n" + "="*70)
print("PERFORMANCE SUMMARY")
print("="*70)

total_time = t_pairs + t_spf_filter + t_spf_detect + t_mdrac_filter + t_mdrac_detect

print(f"\nPair Generation (once): {t_pairs:.2f}s")
print(f"\nSPF Detection:")
print(f"  Filtering: {t_spf_filter:.2f}s")
print(f"  Detection: {t_spf_detect:.2f}s")
print(f"  Subtotal: {t_spf_filter + t_spf_detect:.2f}s")
print(f"\nM-DRAC Detection:")
print(f"  Filtering: {t_mdrac_filter:.2f}s")
print(f"  Detection: {t_mdrac_detect:.2f}s")
print(f"  Subtotal: {t_mdrac_filter + t_mdrac_detect:.2f}s")
print(f"\n{'='*70}")
print(f"TOTAL TIME: {total_time:.2f}s")
print(f"{'='*70}")

print("\n✅ Optimized workflow: Pairs generated ONCE and reused for both detectors")
print(f"   Estimated speedup: ~2x faster compared to generating pairs twice")

# Cleanup base pairs
del base_pairs, spf_pairs, mdrac_pairs
gc.collect()
log_memory("After cleanup")


PERFORMANCE SUMMARY

Pair Generation (once): 2.03s

SPF Detection:
  Filtering: 0.01s
  Detection: 0.12s
  Subtotal: 0.12s

M-DRAC Detection:
  Filtering: 0.01s
  Detection: 0.00s
  Subtotal: 0.01s

TOTAL TIME: 2.17s

✅ Optimized workflow: Pairs generated ONCE and reused for both detectors
   Estimated speedup: ~2x faster compared to generating pairs twice
[MEMORY] After cleanup: 9012.2 MB


9012.203125